## Caribbean Nesting — 18-Day Preview Movie

Visualizes the 3-level Caribbean nesting hierarchy using current output from all three domains.
Style matches the Gibraltar nesting demo Step 7. Full parent domain shown.

| Domain | Resolution | lon | lat | Output source |
|--------|-----------|-----|-----|---------------|
| Parent (carib12) | 1/12° | -98.5→-35.5 | -6→32 | `~/scratch/archive/carib12_nesting_demo` |
| Middle child | 1/48° | -73.5→-62.5 | 10→18 | `~/scratch/archive/carib_mid_child` |
| Tiny child | 1/192° | -70.5→-66.5 | 12→16 | RUNDIR (not yet archived) |

Outputs: `carib_nesting_preview.png`, `carib_nesting_preview.gif`

In [1]:
from pathlib import Path
import subprocess
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import cmocean

ARCHIVE     = Path("~/scratch/archive").expanduser()
TINY_RUNDIR = Path("/glade/derecho/scratch/manishrv/carib_tiny_child/run")

PARENT  = "carib12_nesting_demo"
MID     = "carib_mid_child"
TINY    = "carib_tiny_child"
DOMAINS = [PARENT, MID, TINY]
LABELS  = {PARENT: "1/12°", MID: "1/48°", TINY: "1/192°"}

TRIM     = 2
N_FRAMES = 18

LON_CROP = (-100.0, -33.0)
LAT_CROP = (-7.0, 33.0)

sfc_paths = {
    PARENT: ARCHIVE / PARENT / "ocn/hist" / f"{PARENT}.mom6.h.sfc.2000-01.nc",
    MID:    ARCHIVE / MID    / "ocn/hist" / f"{MID}.mom6.h.sfc.2000-01.nc",
    TINY:   TINY_RUNDIR / f"{TINY}.mom6.h.sfc.2000-01.nc",
}
static_paths = {
    PARENT: ARCHIVE / PARENT / "ocn/hist" / f"{PARENT}.mom6.h.static.nc",
    MID:    ARCHIVE / MID    / "ocn/hist" / f"{MID}.mom6.h.static.nc",
    TINY:   TINY_RUNDIR / f"{TINY}.mom6.h.ocean_geometry.nc",
}

sfc    = {d: xr.open_dataset(sfc_paths[d],    decode_timedelta=False) for d in DOMAINS}
static = {d: xr.open_dataset(static_paths[d])                         for d in DOMAINS}

sl2 = (slice(TRIM, -TRIM), slice(TRIM, -TRIM))
geo = {d: dict(lon=static[d].geolon.values[sl2],
               lat=static[d].geolat.values[sl2],
               wet=static[d].wet.values[sl2]) for d in DOMAINS}

time_start = {PARENT: 2, MID: 0, TINY: 0}

all_speed = {d: [] for d in DOMAINS}
for d in DOMAINS:
    for t in range(N_FRAMES):
        speed = sfc[d].isel(time=time_start[d] + t).speed.values[sl2]
        all_speed[d].append(np.where(geo[d]["wet"] > 0, speed, np.nan))

pooled = np.concatenate([np.stack(all_speed[d]).ravel() for d in DOMAINS])
norm   = Normalize(vmin=0, vmax=np.nanpercentile(pooled, 99))
cmap   = cmocean.cm.speed
dates  = sfc[TINY].time.values[:N_FRAMES]

def bounds(d):
    g = geo[d]
    return np.nanmin(g["lon"]), np.nanmax(g["lon"]), np.nanmin(g["lat"]), np.nanmax(g["lat"])

out_dir   = Path.cwd()
frame_dir = out_dir / "carib_frames"
frame_dir.mkdir(exist_ok=True)

for t in range(N_FRAMES):
    fig, ax = plt.subplots(figsize=(12, 7), facecolor="k")
    ax.set_facecolor("k")

    for i, d in enumerate(DOMAINS):
        g = geo[d]
        z = i * 2
        land = np.where(g["wet"] > 0, np.nan, 1.0)
        ax.pcolormesh(g["lon"], g["lat"], land, cmap="Greys", vmin=0, vmax=1.4,
                      shading="auto", zorder=z)
        ax.pcolormesh(g["lon"], g["lat"], all_speed[d][t], cmap=cmap, norm=norm,
                      shading="auto", zorder=z + 1)

    for d in [MID, TINY]:
        lon0, lon1, lat0, lat1 = bounds(d)
        ax.add_patch(mpatches.Rectangle(
            (lon0, lat0), lon1 - lon0, lat1 - lat0,
            fill=False, edgecolor="white", linewidth=1.2, linestyle="--", zorder=10,
        ))

    ax.text(np.nanmean(geo[MID]["lon"]),  np.nanmax(geo[MID]["lat"])  + 0.3,
            LABELS[MID],  fontsize=8, fontweight="bold", color="white",
            ha="center", va="bottom", zorder=11)
    ax.text(np.nanmean(geo[TINY]["lon"]), np.nanmin(geo[TINY]["lat"]) - 0.25,
            LABELS[TINY], fontsize=8, fontweight="bold", color="white",
            ha="center", va="top",    zorder=11)
    ax.text(LON_CROP[0] + 0.5, LAT_CROP[1] - 0.5, LABELS[PARENT],
            fontsize=9, fontweight="bold", color="white", ha="left", va="top")

    ax.set_xlim(*LON_CROP)
    ax.set_ylim(*LAT_CROP)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(LAT_CROP))))
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
    cb.set_label("surface current speed (m/s)", color="white")
    cb.ax.yaxis.set_tick_params(color="white")
    plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")

    date_str = str(dates[t])[:10]
    ax.set_title(f"Caribbean nested domains — {date_str}", color="white")
    fig.tight_layout()

    fig.savefig(frame_dir / f"frame_{t:03d}.png", dpi=150)
    plt.close(fig)
    print(f"frame {t+1:2d}/{N_FRAMES}  {date_str}")

gif_path = out_dir / "carib_nesting_preview.gif"
result = subprocess.run(
    ["magick", "-delay", "50", "-loop", "0",
     str(frame_dir / "frame_*.png"),
     "-dither", "FloydSteinberg", "-colors", "256",
     str(gif_path)],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"saved {gif_path}")
else:
    print("magick error:", result.stderr)

/glade/derecho/scratch/manishrv/tmp/ipykernel_60155/1697305566.py:37: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior reform date (1582-10-15). To silence this warning specify 'use_cftime=True'.
  static = {d: xr.open_dataset(static_paths[d])                         for d in DOMAINS}


frame  1/18  2000-01-01


frame  2/18  2000-01-02


frame  3/18  2000-01-03


frame  4/18  2000-01-04


...

frame 15/18  2000-01-15


frame 16/18  2000-01-16


frame 17/18  2000-01-17


frame 18/18  2000-01-18


saved /glade/u/home/manishrv/documents/croc/dev/nesting_caribbean_demo/carib_nesting_preview.gif
